In [ ]:
from pathlib import Path
import os

# Detect Colab
IN_COLAB = "COLAB_GPU" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Text_Mining_Project")
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/Text_Mining_Project
RAW_DIR: /content/drive/MyDrive/Text_Mining_Project/data/raw
PROCESSED_DIR: /content/drive/MyDrive/Text_Mining_Project/data/processed


- Import and Setup

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from joblib import dump

df = pd.read_pickle(PROCESSED_DIR / "df_bbc_preprocessed.pkl")

print("Documents loaded:", len(df))
df.head()


Documents loaded: 2232


,text,label,length,clean_text_bow,clean_text_bert,bow_length,bert_length
0,Swap offer for pirated Windows XP\n\nComputer ...,tech,353,swap offer pirated window xp computer giant mi...,Swap offer for pirated Windows XP Computer gia...,212,353
1,DS aims to touch gamers\n\nThe mobile gaming i...,tech,567,ds aim touch gamer mobile gaming industry set ...,DS aims to touch gamers The mobile gaming indu...,318,567
2,Digital guru floats sub-$100 PC\n\nNicholas Ne...,tech,464,digital guru float sub pc nicholas negroponte ...,Digital guru floats sub-$100 PC Nicholas Negro...,240,464
3,The Force is strong in Battlefront\n\nThe warm...,tech,651,force strong battlefront warm reception greet ...,The Force is strong in Battlefront The warm re...,355,651
4,US blogger fired by her airline\n\nA US airlin...,tech,560,us blogger fire airline us airline attendant s...,US blogger fired by her airline A US airline a...,308,560


- Train/Test split for classification

In [ ]:
X_text_bow = df["clean_text_bow"]
y = df["label"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text_bow,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train size:", len(X_train_text))
print("Test size:", len(X_test_text))

Train size: 1785
Test size: 447


- TF-IDF for classification

In [ ]:
tfidf_clf = TfidfVectorizer(
    max_df=0.8,          # ignore frequent terms
    min_df=5,            # ignore rare terms
    ngram_range=(1, 1),
)

X_train_tfidf = tfidf_clf.fit_transform(X_train_text)
X_test_tfidf = tfidf_clf.transform(X_test_text)

print("Shape TF-IDF train:", X_train_tfidf.shape)
print("Shape TF-IDF test:", X_test_tfidf.shape)

Shape TF-IDF train: (1785, 6100)
Shape TF-IDF test: (447, 6100)


- TF-IDF on the whole corpus (for clustering)

In [ ]:
tfidf_all = TfidfVectorizer(
    max_df=0.8,
    min_df=5,
    ngram_range=(1, 1),
)

X_all_tfidf = tfidf_all.fit_transform(df["clean_text_bow"])

print("Shape TF-IDF all docs:", X_all_tfidf.shape)

Shape TF-IDF all docs: (2232, 6985)


- Saving items for later notebooks

In [ ]:
# --- 5.1 Save split information inside the dataframe (optional but ok) ---
df["is_train"] = df.index.isin(X_train_text.index)
df["is_test"]  = df.index.isin(X_test_text.index)

df_out_path = PROCESSED_DIR / "df_bbc_preprocessed_split.pkl"
df.to_pickle(df_out_path)
print("Saved split dataframe to:", df_out_path)

# --- 5.2 Save vectorizers and TF-IDF matrices ---
dump(tfidf_clf, PROCESSED_DIR / "tfidf_vectorizer_clf.joblib")
dump(tfidf_all, PROCESSED_DIR / "tfidf_vectorizer_all.joblib")

dump(X_train_tfidf, PROCESSED_DIR / "X_train_tfidf.joblib")
dump(X_test_tfidf,  PROCESSED_DIR / "X_test_tfidf.joblib")
dump(X_all_tfidf,   PROCESSED_DIR / "X_all_tfidf.joblib")

print("Saved TF-IDF artifacts to:", PROCESSED_DIR)


Saved split dataframe to: /content/drive/MyDrive/Text_Mining_Project/data/processed/df_bbc_preprocessed_split.pkl
Saved TF-IDF artifacts to: /content/drive/MyDrive/Text_Mining_Project/data/processed


- Sentence embeddings for advanced clustering and classification

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm.auto import tqdm

model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

sentences = df["clean_text_bert"].tolist()

embeddings_all = model.encode(
    sentences,
    batch_size=32,
    show_progress_bar=True
)

embeddings_all = np.array(embeddings_all)
print("Shape embeddings_all:", embeddings_all.shape)


emb_path = PROCESSED_DIR / "embeddings_all.npy"
np.save(emb_path, embeddings_all)
print("Saved embeddings to:", emb_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/70 [00:00<?, ?it/s]

Shape embeddings_all: (2232, 384)
Saved embeddings to: /content/drive/MyDrive/Text_Mining_Project/data/processed/embeddings_all.npy
